In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

#### **Documentación de la API de sklearn**
https://scikit-learn.org/stable/api/index.html

**Feature Selection**

https://scikit-learn.org/stable/modules/feature_selection.html

**Feature Extraction**

https://scikit-learn.org/stable/modules/feature_extraction.html

**Model Selection**

https://scikit-learn.org/stable/api/sklearn.model_selection.html

#### **Preparación y exploración inicial del dataset**

Primero se carga el dataset y se separan las variables explicativas (**X**) de la variable objetivo (**y**, la clase del vino).

Luego el dataset se divide en tres subconjuntos:

- **train (60%)**: usado para entrenar el modelo  
- **validation (20%)**: usado para ajustar hiperparámetros  
- **test (20%)**: usado para evaluar el modelo final  

La división se realiza manteniendo la **misma proporción de clases** mediante `stratify`.

Después se visualizan relaciones entre algunas variables mediante un **pairplot**, coloreando los puntos según la clase.

Finalmente se calcula la **proporción de valores faltantes por columna** para detectar posibles problemas de datos incompletos.

In [ ]:
# Carga el dataset
df = pd.read_csv("wine.csv")

# Separa variables (X) y la clase a predecir (y)
X = df.drop("Class", axis=1)
y = df["Class"]

X

In [ ]:

# Train (60%) y temp (40%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# Dividir temp en validation (20%) y test (20%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Visualiza relaciones entre variables, coloreado por clase
sns.pairplot(df, hue="Class", vars=["Alcohol", "Flavanoids", "Proline"])

# Calcula proporción de valores faltantes por columna
df.isna().sum() / len(df)

#### **Funciones Auxiliares**

`evaluar_knn` crea un modelo **KNN** con los parámetros indicados y evalúa su desempeño.

La función:
- crea el modelo KNN
- ajusta el modelo usando los datos de **entrenamiento**
- realiza predicciones sobre **train** y **validation**
- calcula métricas de desempeño (**accuracy** y **balanced accuracy**)

Finalmente devuelve el **modelo** y un **diccionario con los resultados**.

<br>

$accuracy = \frac{\text{correctos}}{\text{total}}$

$recall = \frac{\text{correctos de esa clase}}{\text{total de esa clase}}$

$balanced\ accuracy = \frac{recall_1 + recall_2 + \dots + recall_C}{C}$

<br>

`plot_knn_2d` visualiza el comportamiento de un modelo **KNN** usando solo dos variables del dataset. Nos muestra las fronteras de decisión para cada clase. ¿Qué pasa si modificamos k? 

In [ ]:
def evaluar_knn(X_train, y_train, X_val, y_val, 
                n_neighbors=5, metric='minkowski', p=2, weights='uniform'):
    
    """
    X_train, y_train: datos y etiquetas de entrenamiento.
    X_val, y_val: datos y etiquetas de validación.
    n_neighbors: número de vecinos (k).
    metric: métrica de distancia usada por KNN.
    p: parámetro de Minkowski (p=2 → euclidiana).
    weights: ponderación de vecinos ('uniform' o 'distance').

    Devuelve el modelo entrenado y métricas de desempeño.
    """

    np.random.seed(42)

    modelo = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        metric=metric,
        p=p,
        weights=weights
    )

    modelo.fit(X_train, y_train)

    pred_train = modelo.predict(X_train)
    pred_val = modelo.predict(X_val)

    resultados = {
        "k": n_neighbors,
        "metric": metric,
        "p": p,
        "weights": weights,
        "acc_train": accuracy_score(y_train, pred_train),
        "acc_val": accuracy_score(y_val, pred_val),
        "bacc_train": balanced_accuracy_score(y_train, pred_train),
        "bacc_val": balanced_accuracy_score(y_val, pred_val)
    }

    return modelo, resultados

def plot_knn_2d(X_train, y_train, X_val, y_val, cols, n_neighbors=5):

    """
    X_train, y_train: datos y etiquetas de entrenamiento.
    X_val, y_val: datos y etiquetas de validación.
    cols: lista con dos índices de columnas a usar como ejes.
    n_neighbors: número de vecinos del modelo.

    Grafica los puntos y la frontera de decisión de KNN en 2D.
    """

    # seleccionar columnas por índice
    X_train_2d = X_train[:, cols]
    X_val_2d = X_val[:, cols]

    modelo = KNeighborsClassifier(n_neighbors=n_neighbors)
    modelo.fit(X_train_2d, y_train)

    x_min, x_max = X_train_2d[:,0].min() - 1, X_train_2d[:,0].max() + 1
    y_min, y_max = X_train_2d[:,1].min() - 1, X_train_2d[:,1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = modelo.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7,5))
    plt.contourf(xx, yy, Z, alpha=0.25)

    plt.scatter(
        X_train_2d[:,0],
        X_train_2d[:,1],
        c=y_train,
        edgecolor="k",
        label="train"
    )

    plt.scatter(
        X_val_2d[:,0],
        X_val_2d[:,1],
        c=y_val,
        marker="x",
        s=80,
        label="validation"
    )

    plt.xlabel(f"feature {cols[0]}")
    plt.ylabel(f"feature {cols[1]}")
    plt.title(f"KNN decision boundary (k={n_neighbors})")

    plt.legend()
    plt.show()

**¿Funciona este código de abajo?¿Por qué?**

In [ ]:
#modelo, res = evaluar_knn(X_train,y_train,X_val,y_val,n_neighbors=5)
#res

#### **Imputación de valores faltantes**

En el dataset existen valores faltantes en algunas variables. Antes de entrenar el modelo es necesario tratarlos.

Algunas estrategias posibles son:

- imputar con la **media** de la variable
- imputar con la **mediana**
- imputar con la **moda** (valor más frecuente)
- **eliminar** las filas con valores faltantes

In [ ]:
from sklearn.impute import SimpleImputer

imputer = ...

X_train = ...
X_val = ...
X_test = ...

modelo, res = evaluar_knn(
    X_train,
    y_train,
    X_val,
    y_val,
    n_neighbors=5
)

res

Experimentar graficando las fronteras de decisión con dos features. ¿Qué ocurre al modificar k? ¿Todas las combinaciones de features dividen el espacio de igual forma?

In [ ]:
plot_knn_2d(
    X_train,
    y_train,
    X_val,
    y_val,
    cols=[0, 6],   # features a utilizar
    n_neighbors=5
)

#### **Outliers**
Analizar si existen valores extremos en el dataset. Visualizar con: boxplots, histogramas, scatter plots.

**Preguntas**

1️⃣ ¿Qué variables presentan valores extremos?

2️⃣ ¿Todos los outliers son necesariamente errores?

3️⃣ ¿Cómo podría afectar esto a KNN?


Probar alguna estrategia para eliminar outliers y reentrenar el modelo, ver cómo afectan a la performance de KNN. ¿Es conveniente eliminarlos?

In [ ]:
# visualizar outliers antes

In [ ]:
# Eliminar outliers


In [ ]:
# visualizar outliers después

In [ ]:
modelo, res = evaluar_knn(
    X_train,
    y_train,
    X_val,
    y_val
)

res

#### **Scaling**
Analizar la escala de las variables. Pueden usar `df.describe().T`

**Preguntas**

1️⃣ ¿Las variables tienen escalas similares?

2️⃣ ¿Por qué esto puede ser un problema para KNN?

3️⃣ ¿Qué pasaría si una variable tiene valores mucho más grandes que las demás?


Aplicar distintos scalers: StandardScaler, MinMaxScaler, RobustScaler

In [ ]:
df.describe().T

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = ...

X_train = ...
X_val = ...
X_test = ...

modelo, res = evaluar_knn(
    X_train,
    y_train,
    X_val,
    y_val
)

res

#### **Oversampling y Undersampling**
Examinar la distribución de clases con `y.value_counts()`

**Preguntas**

1️⃣ ¿El dataset está balanceado?

2️⃣ ¿Qué problemas genera el desbalance de clases?
    
3️⃣ ¿Por qué accuracy puede ser engañosa en estos casos?

Probar: Oversampling (Random, SMOTE, ADASYN), Undersampling (Random, NearMiss)


In [ ]:
...

modelo, res = evaluar_knn(
    X_train,
    y_train,
    X_val,
    y_val
)

res

#### **Feature Selection + Extraction + Transformation**

Analizar qué variables parecen más informativas y quitar aquellas que no. Se pueden calcular correlaciones para eliminar variables poco correlacionadas con la clase.

Crear nuevas variables a partir de las anteriores combinandolas o aplicandole fórmulas.

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

selector = SequentialFeatureSelector(
    knn,
    n_features_to_select=5,
    direction="backward"
)

X_train_sel = ...
X_val_sel = ...
X_test_sel = ...

modelo, res = evaluar_knn(
    X_train_sel,
    y_train,
    X_val_sel,
    y_val
)

res


#### **Model Selection (K y distancia)**

KNN depende fuertemente del parámetro K.

**Preguntas**

1️⃣ ¿Qué ocurre cuando k = 1?

2️⃣ ¿Qué ocurre cuando k es muy grande?

3️⃣ ¿Cómo afecta esto al overfitting / underfitting?

Evaluar para distintos valores de **k** y modificar la norma de la distancia. ¿Está bien modificar los hiperparámetros en base a una sola partición de validación?


In [ ]:
ks = range(1, 25)

resultados = []

for k in ks:

    modelo, res = evaluar_knn(
        X_train,
        y_train,
        X_val,
        y_val,
        n_neighbors=k
    )

    resultados.append(res)

resultados

#### **Model Validation**

Utilizar alguna estrategia como LOOCV o k-fold CV para evaluar el modelo y comparar los resultados. Acá pueden probar los variar k y la norma de la distancia. ¿De qué sirve hacer Cross Validation?

In [ ]:
...

#### **Evaluacion Final (Testing)**

In [ ]:
_, res = evaluar_knn(
    X_train,
    y_train,
    X_test,
    y_test,
    n_neighbors=5
)

res